In [1]:
!uv add pandas

Resolved 40 packages in 26.59s
Prepared 2 packages in 2m 01s
Installed 4 packages in 587ms
 + numpy==2.3.3
 + pandas==2.3.2
 + pytz==2025.2
 + tzdata==2025.2


In [39]:
from concurrent.futures import ProcessPoolExecutor as ppe
from generate import generate_csv

NUMBER_OF_FILES = 5
NUMBER_OF_LINES = 1000

if __name__ == "__main__":
    with ppe() as executor:
        futures = [executor.submit(generate_csv,i,NUMBER_OF_LINES) for i in range(1,NUMBER_OF_FILES + 1)]

fn = []
for future in futures:
    fn.append(future.result())
    
futures

[<Future at 0x1ad985bd8d0 state=finished returned str>,
 <Future at 0x1ad985bf450 state=finished returned str>,
 <Future at 0x1ad985bc1d0 state=finished returned str>,
 <Future at 0x1ad985bc350 state=finished returned str>,
 <Future at 0x1ad985bc4d0 state=finished returned str>]

In [40]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor as tpe

def process(path):
    df = pd.read_csv(path)
    return df.groupby('Категория')['Значение'].agg(Медиана = 'median', Стандартное_отклонение='std').reset_index()

with tpe() as executor:
    dataframes = list(executor.map(process,fn))

combined = pd.concat(dataframes, ignore_index=True)

combined

,Категория,Медиана,Стандартное_отклонение
0,A,5235.284473,2700.800658
1,B,4891.557708,2825.562417
2,C,5108.572174,3099.396081
3,D,5299.895882,2910.893637
4,A,5407.372114,2866.128724
5,B,5056.160503,2842.634393
6,C,5506.494267,2754.930086
7,D,5302.603549,2917.466045
8,A,5677.502137,2941.764505
9,B,4848.461342,2922.584178


In [41]:
result = combined.groupby('Категория')['Медиана'].agg(Медиана_из_медиан='median', Стандартное_отклонение_из_медиан='std').reset_index()
result

,Категория,Медиана_из_медиан,Стандартное_отклонение_из_медиан
0,A,5281.644845,195.372081
1,B,5056.160503,220.789570
2,C,5108.572174,472.324387
3,D,5155.954112,284.386324
